# 작업형 1

### 문제 1.

In [122]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/main/p4/12/library_books.csv")
# print(df.shape)
# print(df.head())


# [1-1] 'Python' 또는 '파이썬'이 포함된 도서의 권수 (대소문자 구분 X)
cond1 = df['title'].str.contains("Python", case=False)
cond2 = df['title'].str.contains("파이썬")
print(len(df[cond1 | cond2]))

# [1-2] 대출 가능(available이 'Y')한 도서 중 2020년 이후(2020년 포함)에 출판되고 평점(rating)이 4.0 이상인 도서의 누적 대출횟수(loan_count) 평균
cond1 = (df['available'] == 'Y')
cond2 = (df['pub_year'] >= 2020)
cond3 = (df['rating'] >= 4.0)
df2 = df[cond1 & cond2 & cond3]
print(round(df2['loan_count'].mean(), 3))


141
52.328


### 문제 2.

In [123]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/main/p4/12/delivery.csv")
# print(df.shape)
# print(df.head())
# print(df.isnull().sum())


# 전처리 (1) 평일배달건수와 주말배달건수의 결측치를 각각 해당 컬럼의 평균값으로 대체
df['평일배달건수'] = df['평일배달건수'].fillna(df['평일배달건수'].mean())
df['주말배달건수'] = df['주말배달건수'].fillna(df['주말배달건수'].mean())
# print(df.isnull().sum())

# 전처리 (2) 평일배달건수와 주말배달건수를 더한 상점별 전체배달건수 추가
df['전체배달건수'] = df['평일배달건수'] + df['주말배달건수']
# print(df.head())

# [2-1] 자치구별 전체배달건수의 평균이 가장 높은 자치구와 두 번째로 높은 자치구 + 두 자치구에 속한 상점 수의 합
m = df.groupby('자치구')['전체배달건수'].mean()
top2 = m.sort_values(ascending=False).index[:2]
# print(top2) # 동대문구 / 구로구
# cond = (df['자치구'] == top2[0]) | (df['자치구'] == top2[1])
cond = (df['자치구'] == "구로구") | (df['자치구'] == "동대문구")
print(len(df[cond]))

# [2-2] 문제 2-1에서 찾은 두 자치구에 대하여, 상점별 전체배달건수의 합동분산(Pooled Variance)
# ((자유도1 * 분산1) + (자유도2 * 분산2)) / 자유도1 + 자유도2
cond1 = (df['자치구'] == top2[0])
cond2 = (df['자치구'] == top2[1])

var1 = df[cond1]['전체배달건수'].var() # 분산
var2 = df[cond2]['전체배달건수'].var() # 분산

n1 = sum(cond1)
n2 = sum(cond2)

result = (((n1-1) * var1) + ((n2-1) * var2)) / ((n1-1) + (n2-1))
print(round(result, 2))


39
2112.94


### 문제 3.

In [124]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/main/p4/12/students.csv")
# print(df.shape)
# print(df.head())
# print(df.info())
# print(df.isnull().sum())


# [3-1] 데이터를 복원한 후 전체 학생의 평균 나이
age = df['A'].fillna(df['나이'])
# print(round(age.astype(int).mean(), 3))


# [3-2] 소속별로 복수전공을 하는 학생 수와 복수전공을 하지 않는 학생 수의 차이(절댓값) + 그 중 가장 큰 값
cond = df['A'].notna()

df1 = df[cond] 
df2 = df[~cond]
# print(df1.shape, df2.shape)

cnt1 = df1.groupby('소속').size()
cnt2 = df2.groupby('소속').size()
# print(cnt1, cnt2)

diff = abs(cnt1 - cnt2)
diff = diff.sort_values(ascending=False)
print(diff.max())

72


# 작업형 2

In [125]:
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/main/p4/12/maintenance_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/main/p4/12/maintenance_test.csv")


print(train.shape, test.shape)
# print(train.head())
# print(train.info())
# print(train.isnull().sum())


target = train.pop('maintenance_cost')
print(train.shape, test.shape)


train = train.drop(['id'], axis=1)
test = test.drop(['id'], axis=1)
print(train.shape, test.shape)


num_cols = train.select_dtypes(include='number').columns
cat_cols = train.select_dtypes(exclude='number').columns
print(len(num_cols), len(cat_cols))

for col in cat_cols:
    set_train = set(train[col])
    set_test = set(test[col])
    same = (set_train == set_test)
    if same:
        print(col, "같음")
    else:
        print(col)


train = pd.get_dummies(train)
test = pd.get_dummies(test)
print(train.shape, test.shape)

# n = len(train)
# data = pd.concat([train, test], axis=0)
# data_oh = pd.get_dummies(data)
# train = data_oh.iloc[:n].copy()
# test = data_oh.iloc[n:].copy()
# print(train.shape, test.shape)
print(train.columns.equals(test.columns))


from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=42)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)


from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_log_error

models = {
    "RF": RandomForestRegressor(random_state=42),
    "LGBM": LGBMRegressor(random_state=42, verbose=-1)
}

scores = {}

import numpy as np
for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    pred = np.clip(pred, 0, None)  # 음수를 0으로 변경
    score = root_mean_squared_log_error(y_val, pred)
    scores[name] = score
    print(name, score)

best_name = min(scores, key=scores.get)
best_model = models[best_name]
print(best_name, best_model)

best_model.fit(train, target)
pred = best_model.predict(test)
pred = np.clip(pred, 0, None)

submit = pd.DataFrame({'pred': pred})
submit.to_csv('data/기출12회.csv', index=False)

(5000, 26) (2000, 25)
(5000, 25) (2000, 25)
(5000, 24) (2000, 24)
23 1
equipment_type 같음
(5000, 27) (2000, 27)
True
(4000, 27) (1000, 27) (4000,) (1000,)
RF 0.47873027330969775
LGBM 0.44383632235560566
LGBM LGBMRegressor(random_state=42, verbose=-1)


# 작업형 3

### 문제 1

In [126]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/main/p4/12/exam_scores.csv")
# print(df.shape)
# print(df.head())


# [1-1] 
# 실제 기말고사 점수의 평균이 70점과 같은지 단일표본 t검정(one sample t-test)을 수행하고, p-value를 구하시오.
from scipy import stats
result = stats.ttest_1samp(df['final'], 70)
print(result.statistic, result.pvalue)
print("귀무가설 기각" if result.pvalue < 0.05 else "귀무가설 채택")


# [1-2] 
# 중간고사 점수에 비해 기말고사 점수가 유의하게 향상되었는지 대응표본 t검정(paired t-test)을 수행하고, 검정통계량의 절댓값을 구하시오.
from scipy import stats
result = stats.ttest_rel(df['midterm'], df['final'], alternative='less')
print(abs(result.statistic), result.pvalue)
print("귀무가설 기각" if result.pvalue < 0.05 else "귀무가설 채택")


# [1-3] 
# 학습 프로그램을 수강한 학생과 수강하지 않은 학생의 기말고사 평균 점수 차이(수강 − 미수강)에 대해 두 집단의 분산이 같다고 가정하고 독립표본 t검정을 수행할 때, 평균 차이의 95% 신뢰구간의 하한값을 구하시오.
cond1 = df['program'] == 'yes'
group1 = df[cond1]['final']
cond2 = df['program'] == 'no'
group2 = df[cond2]['final']

result = stats.ttest_ind(group1, group2, equal_var=True)
ci = result.confidence_interval()
print(ci)
print(round(ci.low, 3))

1.4295432250647129 0.1548100466035055
귀무가설 채택
2.142668178313239 0.016830724135690464
귀무가설 기각
ConfidenceInterval(low=np.float64(1.0619946383348329), high=np.float64(8.363005361665179))
1.062


### 문제 2

In [127]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/main/p4/12/course_completion.csv")
# print(df.head())
# print(df.columns)


# [2-1]
# student_id를 제외한 모든 독립 변수를 활용하여 로지스틱 회귀분석을 수행하시오. 
# 과제를 전부 제출하지 않은(assignments_done이 'no') 학생 대비 전부 제출한('yes') 학생의 수료 오즈비(Odds Ratio)
from statsmodels.formula.api import logit
model = logit('completed ~ study_hours + attendance_rate + quiz_score + assignments_done + forum_posts + age', data=df).fit()
# print(model.summary())

import numpy as np
odds_ratio = np.exp(model.params['assignments_done[T.yes]'])
print(round(odds_ratio, 3))


# [2-2]
# 위 모델을 이용하여 아래 조건을 가진 새로운 학생의 수료 확률을 구하시오. 
new = pd.DataFrame([{'study_hours': 8, 'attendance_rate': 80, 'quiz_score': 80,
                     'assignments_done': 'yes', 'forum_posts': 8, 'age': 25}])
pred = model.predict(new)
print(round(pred[0], 3))

# [2-3]
# 위 모델로 전체 데이터의 수료 확률을 예측한 후, 확률이 0.5 이상이면 1(수료), 미만이면 0(미수료)으로 분류했을 때의 정확도(Accuracy)를 구하시오. 
from sklearn.metrics import accuracy_score
pred = model.predict(df)
pred = (pred >= 0.5).astype(int)
acc = accuracy_score(df['completed'], pred)
print(round(acc, 3))

Optimization terminated successfully.
         Current function value: 0.233073
         Iterations 9
86.467
0.995
0.941
